# 07 — Hiểu HGB từ con số 0

## Goal — Một business, một ngày, bao nhiêu Recorded check-ins?

Hãy tưởng tượng bạn theo dõi **hai quán đang hoạt động trong cùng một ngách nhỏ**.
Bạn muốn đọc lịch sử và hiểu một model ước lượng activity như thế nào khi lịch hoặc weather thay đổi.
Notebook này giải thích bằng chuyện, hình và kết quả thật từ notebook 03/05;
không yêu cầu bạn biết lập trình, không huấn luyện lại, không mặc định HGB tốt nhất.

**Recorded check-ins là số hoạt động được ghi nhận trên Yelp.** Nó không phải số khách hàng
hay doanh thu; một người có thể không check-in, hoặc check-in nhiều lần.

## Setup — Nguồn ví dụ và ba con số được phép sửa

Hoàn tất [03](03_location_forecast_poc.ipynb), rồi [05](05_forecast_gam_catboost_comparison.ipynb)
trong môi trường notebook của dự án. Để các đường dẫn override là `None` để đọc cặp run
COMPLETE mới nhất còn xác minh được. Ngách trong câu chuyện được lấy từ cấu hình của run.
`MANUAL_WEATHER` gồm nhiệt độ thấp nhất, cao nhất (°C) và lượng mưa (mm).

Các ô tính toán được đánh dấu **hide-input** để giao diện hỗ trợ có thể thu gọn.
Bạn có thể đọc tiêu đề, bảng và hình mà không cần đọc mã.

In [ ]:
import hashlib
import json
import math
from datetime import UTC, date, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.patches import FancyBboxPatch

from sitesense.business_research import load_models, load_run, predict

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

# Config: None chọn run COMPLETE mới nhất còn xác minh được code/lockfile/nguồn.
# Có thể đặt đường dẫn tuyệt đối hoặc đường dẫn tính từ ROOT để chọn run cụ thể.
COMPARISON_RUN_OVERRIDE = None
POC_RUN_OVERRIDE = None
EXPECTED_SOURCE_CATEGORY = None  # Ví dụ "Pizza"; None đọc đúng ngách của run 03/05.
DEMO_DAY = date(2026, 7, 15)
TOP_N = 8
COMPARE_BUSINESS_IDS = []  # Rỗng: dùng hai business đầu trong bảng kịch bản.
MANUAL_WEATHER = {"tmin_c": 24.0, "tmax_c": 32.0, "precipitation_mm": 20.0}
WEATHER_EXAMPLES = {
    "Mát, khô · giả định": {"tmin_c": 12.0, "tmax_c": 20.0, "precipitation_mm": 0.0},
    "Nóng, khô · giả định": {"tmin_c": 25.0, "tmax_c": 35.0, "precipitation_mm": 0.0},
    "Tự nhập · giả định": MANUAL_WEATHER,
}
SOURCE = ROOT / "notebooks/07_hgb_for_beginners.ipynb"
OUTPUT_ROOT = ROOT / "data/processed/business_hgb_for_beginners"
CALENDAR_REFERENCE_DIR = ROOT / "models/business_forecast"

MODEL_NAMES = {
    "overall_mean": "Trung bình riêng mỗi business",
    "month_weekday_shrunk": "Calendar có co về trung bình",
    "poisson": "Poisson regression + weather",
    "hgb": "HGB + weather",
    "gam_poisson": "GAM dạng spline + weather",
    "catboost": "CatBoost + weather",
}
CALENDAR_MODELS = ["overall_mean", "month_weekday_shrunk"]
WEATHER_MODELS = ["poisson", "hgb", "gam_poisson", "catboost"]
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def file_hashes(directory):
    return {
        str(p.relative_to(directory)): sha256(p)
        for p in sorted(directory.rglob("*"))
        if p.is_file()
    }


def configured_path(value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()


def load_compatible_evidence():
    if COMPARISON_RUN_OVERRIDE is not None:
        candidates = [configured_path(COMPARISON_RUN_OVERRIDE)]
    else:
        candidates = sorted(
            (ROOT / "data/processed/business_model_comparison").glob("*/manifest.json"),
            reverse=True,
        )
    failures = []
    for path in candidates:
        try:
            comparison = load_run(path.parent if path.name == "manifest.json" else path)
            parent = comparison.manifest.get("parent_run")
            if not parent:
                raise ValueError("Run 05 thiếu parent_run từ notebook 03")
            parent_path = configured_path(parent["directory"])
            if POC_RUN_OVERRIDE is not None and parent_path != configured_path(POC_RUN_OVERRIDE):
                raise ValueError("Run 05 không bắt nguồn từ POC_RUN_OVERRIDE")
            poc = load_run(parent_path)
            if sha256(parent_path / "manifest.json") != parent["manifest_sha256"]:
                raise ValueError("Hash manifest cha 03 đã thay đổi")
            for key in [
                "grain",
                "config",
                "protocol",
                "source_signature",
                "catalog_signature",
                "panel_signatures",
            ]:
                if poc.manifest[key] != comparison.manifest[key]:
                    raise ValueError(f"Run 03/05 không cùng {key}")
            if not set(MODEL_NAMES).issubset(comparison.manifest["model_names"]):
                raise ValueError("Run 05 chưa chứa đủ sáu model")
            if EXPECTED_SOURCE_CATEGORY is not None:
                if comparison.manifest["config"]["source_category"] != EXPECTED_SOURCE_CATEGORY:
                    raise ValueError("Run có source category khác cấu hình mong muốn")
            return poc, comparison, failures
        except (OSError, ValueError, KeyError, RuntimeError) as error:
            if COMPARISON_RUN_OVERRIDE is not None:
                raise
            failures.append({"run": path.name, "reason": str(error)[:240]})
    raise RuntimeError(
        "Chưa có cặp run 03/05 COMPLETE tương thích. Chạy lại 03 rồi 05. " + str(failures[:3])
    )


poc, comparison, skipped_runs = load_compatible_evidence()
metrics = comparison.metrics.copy()
config = comparison.manifest["config"]
models = load_models(comparison.directory)
hgb = models["hgb"].estimator
parameters = hgb.get_params(deep=False)
assert {key: str(value) for key, value in parameters.items()} == {
    key: str(value) for key, value in models["hgb"].parameters.items()
}
calendar_reference_before = file_hashes(CALENDAR_REFERENCE_DIR)
RUN_DIR = OUTPUT_ROOT / ("run_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ"))
(RUN_DIR / "figures").mkdir(parents=True)
(SOURCE_COPY := RUN_DIR / "notebook_source.ipynb").write_bytes(SOURCE.read_bytes())

matched = metrics.loc[metrics["population"].eq("accepted_weather")].copy()
validation = matched.loc[matched["stage"].eq("validation")].set_index("model")
assert not validation.index.duplicated().any()
assert comparison.manifest["selection_frozen_before_test"]
for stage in ["validation", "test"]:
    same_rows = matched.loc[matched["stage"].eq(stage)]
    for column in ["businesses", "observations", "recorded_checkins"]:
        assert same_rows[column].nunique() == 1, f"Mẫu chấm khác nhau: {stage}/{column}"
selection_order = validation.reindex(comparison.manifest["model_names"]).sort_values(
    "poisson_deviance", kind="stable"
)
selected_model = comparison.manifest["selected_model"]
assert selected_model == selection_order.index[0]
best_calendar = (
    validation.loc[CALENDAR_MODELS].sort_values("poisson_deviance", kind="stable").index[0]
)
best_weather = (
    validation.loc[WEATHER_MODELS].sort_values("poisson_deviance", kind="stable").index[0]
)
print(
    "Ngách:",
    config["city"],
    config["state"],
    "→",
    config["category"],
    "→",
    config["source_category"],
)
print("03:", poc.directory.relative_to(ROOT))
print("05:", comparison.directory.relative_to(ROOT))
if skipped_runs:
    print(
        f"Đã bỏ qua {len(skipped_runs)} run cũ/không tương thích; không ghép metrics từ các run đó."
    )

# Artifact/source đã được load_run/load_models xác minh. Chỉ suy luận bằng model đã nạp;
# cuối notebook xác minh đầy đủ lại trước khi ghi manifest COMPLETE.
verified_profiles = pd.DataFrame(json.loads((comparison.directory / "catalog.json").read_text()))
verified_profiles = verified_profiles.rename(columns={"categories": "source_categories"})


def scenario_predictions(model_name, weather=None, top_n=10):
    if DEMO_DAY <= date.fromisoformat(comparison.manifest["protocol"]["final"][1]):
        raise ValueError("Kịch bản phải sau cửa sổ final; đọc held-out predictions cho lịch sử")
    fitted = models[model_name]
    frame = pd.DataFrame({"business_id": fitted.business_ids, "day": DEMO_DAY})
    frame["month"], frame["weekday"] = DEMO_DAY.month, DEMO_DAY.weekday()
    if fitted.weather_features:
        if weather is None or set(weather) != {"tmin_c", "tmax_c", "precipitation_mm"}:
            raise ValueError("Cần kịch bản đầy đủ nhiệt độ °C và lượng mưa mm")
        if not all(math.isfinite(value) for value in weather.values()):
            raise ValueError("Các giá trị weather phải hữu hạn")
        if weather["tmin_c"] > weather["tmax_c"] or weather["precipitation_mm"] < 0:
            raise ValueError("Kịch bản weather không nhất quán")
        for column, value in weather.items():
            frame[column] = value
    frame["weather_available"] = True
    frame["predicted_checkins"] = predict(fitted, frame)
    frame = frame.merge(
        verified_profiles[["business_id", "name", "source_categories", "latitude", "longitude"]],
        on="business_id",
        how="left",
        validate="one_to_one",
    )
    return frame.sort_values(["predicted_checkins", "business_id"], ascending=[False, True]).head(
        top_n
    )

In [ ]:
display(
    Markdown(
        f"**Câu chuyện hôm nay:** theo dõi các business thuộc "
        f"**{config['source_category']}** trong **{config['category']}**, "
        f"tại **{config['city']}, {config['state']}**."
    )
)

## Steps

### 1. Nhìn đúng một hàng dữ liệu

| Mảnh thông tin | Ví dụ ý nghĩa |
| --- | --- |
| Business | Một listing cụ thể có `business_id` |
| Ngày | Một ngày thuộc coverage đã xác nhận |
| Lịch | Tháng nào, thứ mấy |
| Weather | Nhiệt độ và mưa được nguồn chấp nhận |
| Kết quả thật | Tổng Recorded check-ins của business trong ngày đó |

Không có sự kiện trong một ngày **đã covered** là số 0 có ý nghĩa.
Ngày ngoài coverage hoặc đọc nguồn thất bại là **chưa biết**, không phải số 0.
Một business có thể gắn nhiều Source Category; đếm lại ở nhiều ngách không làm nó
thành nhiều business khác nhau. Chúng ta chọn đúng city/state và đúng nhãn nguồn.

### 2. “Học” và “dự đoán” khác nhau thế nào?

Khi học, máy nhìn các ví dụ cũ **kèm đáp án check-in thật** để tìm quy tắc.
Khi dự đoán một ngày mới, máy dùng business đã biết, lịch và weather giả định;
không cần nhập số check-in thật của ngày chưa xảy ra.

Model có thể học mức activity riêng của từng business. Vì thế, ví dụ này nói về
**business đã có trong cohort training**, không tự áp dụng được cho một quán mới chưa có lịch sử.
Đánh giá sao, review count và trạng thái mở cửa từ snapshot hiện tại không được đưa vào
để giả vờ rằng model đã biết các thông tin đó ở quá khứ.

### 3. Một cây hỏi vài câu nhỏ

Cây quyết định chia các ví dụ thành nhóm bằng các câu hỏi. Hình dưới là **minh họa cách nghĩ**,
không phải cây hay ngưỡng thật trích từ HGB. Tên và quy tắc trong hình không phải khuyến nghị kinh doanh.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.set(xlim=(0, 10), ylim=(0, 4.2))
ax.axis("off")
boxes = [
    (3.4, 2.9, 3.2, 0.8, "Hôm nay là cuối tuần?", "#e5eef8"),
    (0.5, 0.8, 3.5, 1.0, "Nhóm A\nHọc từ ví dụ ngày thường", "#f6edd5"),
    (6.0, 0.8, 3.5, 1.0, "Nhóm B\nHọc từ ví dụ cuối tuần", "#e2efe8"),
]
for x, y, width, height, text, color in boxes:
    ax.add_patch(
        FancyBboxPatch(
            (x, y), width, height, boxstyle="round,pad=0.08", facecolor=color, edgecolor="#bbc7ca"
        )
    )
    ax.text(x + width / 2, y + height / 2, text, ha="center", va="center")
for end, label, label_x in [((2.3, 1.9), "Không", 2.7), ((7.7, 1.9), "Có", 7.0)]:
    ax.annotate("", xy=end, xytext=(5, 2.85), arrowprops={"arrowstyle": "->"})
    ax.text(label_x, 2.3, label, ha="center")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/a_small_tree.png", dpi=150)
plt.show()

### 4. Nhiều cây cùng sửa lỗi từng chút

**HGB = Histogram-based Gradient Boosting.** Có thể hiểu đơn giản là:

1. Bắt đầu bằng một ước lượng đơn giản từ training.
2. Học một cây nhỏ để sửa phần lỗi còn lại.
3. Thêm nhiều cây, mỗi lần chỉ sửa một ít.

“Histogram” giúp gộp giá trị thành các khoảng để tìm điểm chia hiệu quả.
“Poisson loss” là quy tắc chấm lỗi phù hợp với số đếm không âm; nó không biến HGB thành
Poisson regression. Các model có thể cùng mục tiêu chấm lỗi mà học theo những cách khác nhau.

Model quá đơn giản có thể **học thiếu**; model nhớ cả nhiễu có thể **quá khớp**.
Chấm các ngày tương lai giúp xem những quy tắc đã học có còn hữu ích không.

### 5. Ba “nút điều chỉnh” dễ nhớ

Giá trị trong bảng lấy từ **HGB final đã lưu của run 05**. Notebook này chỉ đọc model;
không tăng giảm tham số hoặc chạy lại một phép huấn luyện nặng.
Không có nút nào cứ tăng lên là luôn tốt hơn.

In [ ]:
three_knobs = pd.DataFrame(
    [
        {
            "Nút": "max_iter — số vòng học",
            "Giá trị thật": str(parameters["max_iter"]),
            "Hiểu đơn giản": "Cho phép sửa lỗi nhiều vòng hơn; có thể tốn thời gian và học nhiễu.",
        },
        {
            "Nút": "learning_rate — mức sửa mỗi vòng",
            "Giá trị thật": str(parameters["learning_rate"]),
            "Hiểu đơn giản": "Sửa nhẹ thường cần nhiều vòng hơn; phải xem cùng max_iter.",
        },
        {
            "Nút": "max_leaf_nodes — độ chi tiết mỗi cây",
            "Giá trị thật": str(parameters["max_leaf_nodes"]),
            "Hiểu đơn giản": "Nhiều lá cho quy tắc chi tiết hơn, nhưng dễ ghi nhớ dữ liệu cũ.",
        },
    ]
)
display(three_knobs)
three_knobs.to_csv(RUN_DIR / "three_knobs.csv", index=False)

### 6. Làm sao biết một cách dự đoán có tốt không?

| Lần thử | Học từ | Chấm trên | Dùng để làm gì? |
| --- | --- | --- | --- |
| Validation | 2017–2019 | 2020 | Chọn model |
| Test | 2018–2020 | 2021 | Đọc kết quả sau khi đã chọn |
| Final | 2019–2021 | Chưa có đáp án tương lai | Tạo ví dụ dự đoán |

Mỗi lần, chọn cohort bằng **training** rồi giữ nguyên business khi chấm.
Không dùng việc một business có check-in ở tương lai để quyết định đưa nó vào quá khứ.

Calendar baseline là cách đoán theo lịch sử riêng của business, tháng và thứ, **không cần weather**.
Để so công bằng, bảng dưới chấm tất cả model trên **cùng các ngày đủ weather**.
Deviance và MAE đều càng thấp càng tốt; MAE có thể đọc gần như sai số check-in trung bình mỗi ngày.
Đây là backtest thăm dò trên các năm đã được xem, không phải chứng nhận accuracy cho 2026.

**Một giới hạn của phép so:** calendar học từ toàn bộ ngày covered và pool category cha,
trong khi các model weather học từ những ngày đủ accepted weather. Chấm trên cùng ngày
không xóa được khác biệt training này. Vì thế, điểm tốt hơn chưa chứng minh chỉ riêng việc
thêm weather đã làm model tốt hơn; cần một phép thử giữ nguyên training để tách riêng điều đó.

### Mẫu weather bao phủ bao nhiêu lịch sử?

Tỷ lệ dưới đây là **business-day đánh giá đủ accepted weather / toàn bộ business-day covered**,
không phải tỷ lệ listing có dữ liệu và không phải completeness trung bình đều của các trạm.
Nhiều business có thể dùng chung một trạm; thiếu báo cáo trạm có thể loại đồng thời nhiều listing
và nhiều ngày của cùng mùa. Không giả định các ngày thiếu weather được chọn ngẫu nhiên.
**Kết quả matched-weather chỉ đại diện các dòng còn lại, không phải activity đầy đủ cả năm.**

In [ ]:
weather_coverage = pd.DataFrame(
    [
        {
            "stage": stage,
            "covered_business_days": counts["assessment_days"],
            "accepted_weather_business_days": counts["accepted_assessment_weather_days"],
            "weather_coverage_pct": 100
            * counts["accepted_assessment_weather_days"]
            / counts["assessment_days"],
        }
        for stage, counts in comparison.manifest["stages"].items()
        if stage in ["validation", "test"]
    ]
)
display(
    weather_coverage.rename(
        columns={
            "stage": "Giai đoạn",
            "covered_business_days": "Business-day covered",
            "accepted_weather_business_days": "Business-day đủ weather",
            "weather_coverage_pct": "Weather coverage (%)",
        }
    ).round(2)
)
weather_coverage.to_csv(RUN_DIR / "weather_coverage.csv", index=False)

In [ ]:
simple_scores = (
    matched.pivot(index="model", columns="stage", values="poisson_deviance")
    .reindex(MODEL_NAMES)
    .rename(index=MODEL_NAMES)
    .rename(columns={"validation": "Validation 2020", "test": "Test 2021"})
)
display(simple_scores.round(5))
simple_scores.to_csv(RUN_DIR / "simple_scores.csv", index_label="Model")

### 7. HGB có thắng không? Hãy đọc số liệu

Không chọn HGB chỉ vì tên nghe phức tạp. Dòng dưới gọi tên model được run 05 chọn
**từ validation**, rồi đọc test của chính model đó. Nếu HGB không thắng, nó vẫn là ví dụ
hữu ích để học cách các cây phối hợp. Kết quả này không tự thay artifact phục vụ app. HGB serving hỗ trợ weather scenario theo chính sách sản phẩm;
đó là lựa chọn về khả năng sản phẩm, không phải bằng chứng HGB thắng về accuracy.

In [ ]:
test_scores = matched.loc[matched["stage"].eq("test")].set_index("model")
display(
    Markdown(
        f"**Model được chọn: {MODEL_NAMES[selected_model]}.** "
        f"Deviance validation: **{validation.loc[selected_model, 'poisson_deviance']:.5f}**; "
        f"test: **{test_scores.loc[selected_model, 'poisson_deviance']:.5f}**. "
        f"HGB có validation **{validation.loc['hgb', 'poisson_deviance']:.5f}** "
        f"và test **{test_scores.loc['hgb', 'poisson_deviance']:.5f}**."
    )
)

Calendar còn dự đoán được cho ngày thiếu weather, nếu lịch sử business được hỗ trợ.
Bảng nhỏ dưới chấm calendar trên **toàn bộ ngày covered**. Đó là mẫu khác;
không lấy chênh lệch với bảng trên để tuyên bố weather giúp tăng độ chính xác.

In [ ]:
calendar_all = metrics.loc[
    metrics["population"].eq("all_covered"),
    [
        "stage",
        "model",
        "businesses",
        "observations",
        "poisson_deviance",
        "mae",
        "recorded_checkins",
        "predicted_checkins",
    ],
].copy()
display(calendar_all.round(5))
calendar_all.to_csv(RUN_DIR / "calendar_all_covered.csv", index=False)

### 8. Gặp hai business trong đúng ngách này

Để chọn ví dụ có thật, chúng ta xếp các business **đã có trong cohort final** bằng một
kịch bản HGB cùng ngày/weather, rồi lấy hai dòng đầu có lịch sử test đã lưu. Cohort final và test có thể khác nhau;
không có lịch sử của một ID trong test không được vẽ thành số 0. Đây là danh sách minh họa trong mẫu nghiên cứu,
không phải top toàn bộ thị trường. Vị trí là tọa độ của listing đã có, không phải địa điểm mở quán mới.

In [ ]:
assert MANUAL_WEATHER["tmin_c"] <= MANUAL_WEATHER["tmax_c"]
assert MANUAL_WEATHER["precipitation_mm"] >= 0
scenario_ranking = scenario_predictions("hgb", weather=MANUAL_WEATHER, top_n=TOP_N)
assert not scenario_ranking.empty
assert scenario_ranking["business_id"].is_unique
calendar_ranking = scenario_predictions(best_calendar, top_n=100_000)
scenario_table = scenario_ranking.merge(
    calendar_ranking[["business_id", "predicted_checkins"]].rename(
        columns={"predicted_checkins": "calendar_checkins"}
    ),
    on="business_id",
    how="left",
    validate="one_to_one",
)
assert scenario_table["calendar_checkins"].notna().all()
display(
    scenario_table[
        [
            "business_id",
            "name",
            "source_categories",
            "predicted_checkins",
            "calendar_checkins",
            "latitude",
            "longitude",
        ]
    ].round(3)
)
scenario_table.to_csv(RUN_DIR / "business_scenario_ranking.csv", index=False)

In [ ]:
test_history_ids = set(
    comparison.predictions.loc[
        comparison.predictions["stage"].eq("test")
        & comparison.predictions["model"].eq("month_weekday_shrunk")
        & comparison.predictions["population"].eq("all_covered"),
        "business_id",
    ]
)
chosen_ids = (
    COMPARE_BUSINESS_IDS
    or scenario_table.loc[scenario_table["business_id"].isin(test_history_ids), "business_id"]
    .head(2)
    .tolist()
)
if not set(chosen_ids).issubset(test_history_ids):
    raise ValueError("Business không thuộc cohort test; chọn ID có lịch sử test đã lưu")
if not 1 <= len(chosen_ids) <= 2 or len(set(chosen_ids)) != len(chosen_ids):
    raise ValueError("Chọn một hoặc hai business_id khác nhau")
known_businesses = calendar_ranking.set_index("business_id")
if not set(chosen_ids).issubset(known_businesses.index):
    raise ValueError("Business ngoài cohort final đã fit; không suy diễn sang listing mới")
profiles = known_businesses.loc[chosen_ids, ["name", "source_categories", "latitude", "longitude"]]
display(profiles)
recorded = comparison.predictions.loc[
    comparison.predictions["stage"].eq("test")
    & comparison.predictions["model"].eq("month_weekday_shrunk")
    & comparison.predictions["population"].eq("all_covered")
    & comparison.predictions["business_id"].isin(chosen_ids),
    ["business_id", "day", "recorded_checkins"],
].copy()
recorded["month"] = pd.to_datetime(recorded["day"]).dt.to_period("M").dt.to_timestamp()
assert set(chosen_ids).issubset(set(recorded["business_id"]))
monthly = recorded.groupby(["month", "business_id"])["recorded_checkins"].sum().unstack()
fig, ax = plt.subplots(figsize=(11, 3.8))
for business_id in chosen_ids:
    if business_id in monthly:
        ax.plot(
            monthly.index,
            monthly[business_id],
            marker="o",
            label=f"{known_businesses.loc[business_id, 'name']} · {business_id[:8]}",
        )
ax.set(title="Business trong cùng ngách · test 2021", ylabel="Recorded check-ins / tháng")
ax.legend(fontsize=9)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/two_businesses_recorded.png", dpi=150)
plt.show()
monthly.to_csv(RUN_DIR / "two_businesses_monthly.csv")

Biểu đồ cộng Recorded check-ins theo **cùng tháng của 2021** cho từng business.
Business nhiều check-in hơn chỉ có activity được ghi nhận nhiều hơn trong mẫu;
chưa thể kết luận nhiều khách, doanh thu cao hơn hay vận hành tốt hơn.

### 9. Tự thử weather năm 2026

Giữ nguyên **một business đã biết** và một ngày tương lai, chỉ thay ba con số weather.
Ví dụ dưới dùng `MANUAL_WEATHER` ở đầu notebook cùng hai kịch bản minh họa.
Nhiệt độ thấp nhất phải ≤ cao nhất; lượng mưa không âm. Đơn vị là °C và mm.

Cột HGB có thể thay đổi; calendar giữ nguyên vì không dùng weather.
Số đếm dự đoán có thể là số thập phân: đó là kỳ vọng, không phải lời hứa có đúng từng ấy check-in.

In [ ]:
demo_business_id = chosen_ids[0]
scenario_rows = []
for label, weather in WEATHER_EXAMPLES.items():
    row = (
        scenario_predictions("hgb", weather=weather, top_n=100_000)
        .set_index("business_id")
        .loc[demo_business_id]
    )
    scenario_rows.append(
        {
            "Kịch bản": label,
            **weather,
            "HGB check-ins": row["predicted_checkins"],
            "Calendar check-ins": known_businesses.loc[demo_business_id, "predicted_checkins"],
        }
    )
weather_scenarios = pd.DataFrame(scenario_rows)
display(
    Markdown(
        f"**Business giữ cố định:** {known_businesses.loc[demo_business_id, 'name']} "
        f"(`{demo_business_id}`) · **ngày giả định:** {DEMO_DAY}."
    )
)
display(weather_scenarios.round(3))
fig, ax = plt.subplots(figsize=(10, 3.8))
weather_scenarios.set_index("Kịch bản")[["HGB check-ins", "Calendar check-ins"]].plot.bar(
    ax=ax, color=["#317b63", "#d6ad52"], rot=0
)
ax.set(ylabel="Projected recorded check-ins / business / ngày", xlabel="")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/weather_scenarios.png", dpi=150)
plt.show()
weather_scenarios.to_csv(RUN_DIR / "weather_scenarios.csv", index=False)

## Checks — Năm câu hỏi tự kiểm tra

1. **Có phải nhập check-in thật để dự đoán ngày mới?** Không; đáp án thật chỉ dùng để học và chấm.
2. **Một ngày không có dữ liệu có luôn là 0?** Không; chỉ ngày đã xác nhận coverage và không có event mới là 0.
3. **HGB có luôn thắng calendar?** Không; đọc validation của cùng mẫu và giữ test ngoài lựa chọn.
4. **Mưa làm dự đoán giảm có chứng minh mưa làm khách giảm?** Không; đó là quan hệ thống kê trong model.
5. **Có thể dùng ví dụ này để bảo đảm quán mới sẽ thành công năm 2026?** Không; model nhận identity
   của business đã học, weather là giả định, không có nhãn 2026 để chấm và check-in không phải doanh thu.

Ô cuối xác nhận calendar reference không đổi và lưu lại hash hai run cha để người khác kiểm tra.

## Next Steps — Kết luận của chính lần chạy này

Kết luận được tính từ metrics đã nạp; không có con số hay tên model thắng viết sẵn.

In [ ]:
heldout = matched.loc[matched["stage"].eq("test")].set_index("model")
weather_validation = float(validation.loc[best_weather, "poisson_deviance"])
calendar_validation = float(validation.loc[best_calendar, "poisson_deviance"])
relative_change = (
    (weather_validation / calendar_validation - 1) * 100 if calendar_validation else None
)
comparison_phrase = (
    f"cao hơn {relative_change:.2f}%"
    if relative_change is not None and relative_change >= 0
    else f"thấp hơn {-relative_change:.2f}%"
    if relative_change is not None
    else "không tính tỷ lệ vì calendar deviance bằng 0"
)
selected_validation = float(validation.loc[selected_model, "poisson_deviance"])
selected_test = float(heldout.loc[selected_model, "poisson_deviance"])
coverage_lookup = weather_coverage.set_index("stage")["weather_coverage_pct"]
summary = (
    f"Trên ngách {config['source_category']} tại {config['city']}, {config['state']}, "
    f"run 05 chọn **{MODEL_NAMES[selected_model]}** từ validation 2020. "
    f"Model này có deviance validation **{selected_validation:.5f}** "
    f"và test 2021 **{selected_test:.5f}** trên cùng tập ngày đủ weather. "
    f"Nhánh weather tốt nhất theo validation là **{MODEL_NAMES[best_weather]}**; "
    f"deviance {comparison_phrase} so với calendar tốt nhất trên đúng các dòng đó. "
    "Đây là kết quả của cohort/config đã lưu, không chứng minh weather gây ra check-in "
    "hoặc model thắng ở ngách khác. Training calendar và weather khác tập dòng, nên chênh lệch "
    "điểm chưa cô lập lợi ích riêng của weather. HGB được giải thích để hiểu mô hình; "
    "notebook không "
    "ghi đè artifact phục vụ ứng dụng. HGB serving hỗ trợ weather scenario "
    "theo chính sách sản phẩm; "
    "điều đó không có nghĩa HGB thắng accuracy. "
    f"Weather coverage của validation là {coverage_lookup.loc['validation']:.2f}% "
    f"và test là {coverage_lookup.loc['test']:.2f}% business-day covered; "
    "matched-weather metrics không đại diện activity đầy đủ cả năm."
)
display(Markdown(summary))
(RUN_DIR / "summary.md").write_text(summary + "\n", encoding="utf-8")
assert calendar_reference_before == file_hashes(CALENDAR_REFERENCE_DIR), (
    "Không được sửa calendar reference"
)
reverified = load_run(comparison.directory)
assert reverified.manifest == comparison.manifest
manifest = {
    "status": "complete",
    "schema_version": 1,
    "grain": "business_id_calendar_day",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "purpose": "Vietnamese beginner tutorial using verified business experiments; no retraining",
    "config": config,
    "protocol": comparison.manifest["protocol"],
    "selected_model": selected_model,
    "best_validation_weather_model": best_weather,
    "parents": [
        {
            "directory": str(run.directory.resolve()),
            "manifest_sha256": sha256(run.directory / "manifest.json"),
        }
        for run in [poc, comparison]
    ],
    "source_signature": comparison.manifest["source_signature"],
    "catalog_signature": comparison.manifest["catalog_signature"],
    "panel_signatures": comparison.manifest["panel_signatures"],
    "notebook_source_sha256": sha256(SOURCE_COPY),
    "lockfile_sha256": sha256(ROOT / "uv.lock"),
    "model_parameters": {key: str(value) for key, value in parameters.items()},
    "calendar_reference_unchanged_sha256": calendar_reference_before,
    "demo": {
        "business_id": demo_business_id,
        "day": DEMO_DAY.isoformat(),
        "manual_weather": MANUAL_WEATHER,
        "interpretation": "conditional_existing_business_only",
    },
    "output_sha256": file_hashes(RUN_DIR),
}
(RUN_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã lưu:", RUN_DIR.relative_to(ROOT))

### Từ điển nhỏ và nơi đọc tiếp

| Từ | Hiểu ngắn gọn |
| --- | --- |
| Business identity | Mã listing; chỉ các listing đã học mới được ví dụ hỗ trợ |
| Source Category | Nhãn ngách cụ thể từ nguồn, có thể chồng lặp |
| Cohort | Tập business được chọn bằng dữ liệu training rồi giữ cố định khi chấm |
| Model | Quy tắc dự đoán học từ ví dụ |
| Feature | Thông tin đầu vào của model |
| Target | Recorded check-ins thật của business trong một ngày |
| Calendar baseline | Mốc dự đoán từ lịch, không dùng weather |
| Validation / test | Tập chọn model / tập đọc kết quả sau lựa chọn |
| Kịch bản có điều kiện | “Nếu weather như thế này” tại business đã biết, không phải lời hứa tương lai |

- [06 — Báo cáo phân tích và toàn bộ tham số HGB](06_hgb_model_selection_report.ipynb)
- [03 — Chuẩn bị thí nghiệm business-day](03_location_forecast_poc.ipynb)
- [05 — So sánh các model](05_forecast_gam_catboost_comparison.ipynb)
- [Cách dùng ứng dụng hiện tại](../docs/business-location-workspace.md)